In [1]:
import pandas as pd
from ortools.sat.python import cp_model

In [2]:
people_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main People')
jobs_df = pd.read_excel('data_set/Optimised Data.xlsx', sheet_name='Main Desk')

# jobs_df
# people_df

In [ ]:
# map all the value 
grade_map = {'A':1, 'B':2, 'C':3, 'D':4, 'F':5}
edu_map = {'Doc': 1, 'Degree':2, 'Uni':2, 'Dipolma':3, 'O Level':4, 'N Level':5}
health_map = {'Fit':1, 'Semi Fit':2, 'Not Fit': 3}
sec_map = {'CAT1':1, 'CAT2':2, 'CAT3':3, 'CAT4':4, 'CAT5':5, 'CAT6':6, 'CAT7':7, 'CAT8':8, 'CAT9':9, 'CAT10':10}

jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Desk_ID          942 non-null    object
 1   Req_Grade        942 non-null    object
 2   Req_Edu          942 non-null    object
 3   Req_Health       942 non-null    object
 4   Sec_Clerance     942 non-null    object
 5   Req_Appointment  942 non-null    object
dtypes: object(6)
memory usage: 44.3+ KB


In [ ]:
people_df['Grade'] = people_df['Grade'].replace(grade_map)
people_df['Edu_Type'] = people_df['Edu_Type'].replace(edu_map)
people_df['Health'] = people_df['Health'].replace(health_map)
people_df['Security'] = people_df['Security'].replace(sec_map)

jobs_df['Req_Grade'] = jobs_df['Req_Grade'].replace(grade_map)
jobs_df['Req_Edu'] = jobs_df['Req_Edu'].replace(edu_map)
jobs_df['Req_Health'] = jobs_df['Req_Health'].replace(health_map)
jobs_df['Sec_Clerance'] = jobs_df['Sec_Clerance'].replace(sec_map)


C:\Users\Cherry\AppData\Local\Temp\ipykernel_11404\3982109371.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  people_df['Grade'] = people_df['Grade'].replace(grade_map)
C:\Users\Cherry\AppData\Local\Temp\ipykernel_11404\3982109371.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  people_df['Edu_Type'] = people_df['Edu_Type'].replace(edu_map)
C:\Users\Cherry\AppData\Local\Temp\ipykernel_11404\3982109371.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the 

In [ ]:
appointment_cols = ["Appointments_1", "Appointments_2", "Appointments_3", "Appointments _4"]

# Get all the appointment and put into 1 column
people_df["Appointments"] = (
    people_df[appointment_cols]
    .apply(lambda row: [a for a in row if pd.notna(a) and a != ""], axis=1)
)
people_df = people_df.drop(columns=appointment_cols)

people_df

,Name_ID,Name,Grade,Edu_Type,Health,Security,Appointments
0,P000001,Roxuqy Tiwoni,2,2,2,5,"[Business Development, Customer Support, Custo..."
1,P000002,Lyxu Subyzu,5,3,1,3,"[Logistics, Risk Management, Supply Chain, Ris..."
2,P000003,Cagyla Gedyzo,4,1,2,10,"[Corporate Affairs, Human Resources (HR), Inve..."
3,P000004,Huwyle Wola,3,2,3,7,"[Corporate Affairs, Internal Audit, Sales, Inv..."
4,P000005,Levyjo Buti,1,1,1,10,"[Facilities Management, Maintenance, Health & ..."
...,...,...,...,...,...,...,...
1992,P001993,Mozuni Ciqe,5,2,1,4,"[Quality Assurance (QA), Operations, Complianc..."
1993,P001994,Vogu Sysu,4,5,2,1,"[Project Management Office (PMO), Internal Aud..."
1994,P001995,Tybufe Cotysy,1,5,1,6,"[Health & Safety, Partnerships & Alliances, Go..."
1995,P001996,Viji Zuqemi,4,3,1,7,"[Corporate Affairs, Media Production, Public R..."


In [6]:
print("Starting the model")

Starting the model


In [ ]:
model = cp_model.CpModel()
assign = {}

# Getting all people who matches the job base on hard constraints
for p_idx, person in people_df.iterrows():
    for j_idx, job in jobs_df.iterrows():
        if person['Health'] != job['Req_Health']:
            continue
        if person['Security'] != job['Sec_Clerance']:
            continue
        var = model.NewBoolVar(f"assign_p{p_idx}_j{j_idx}")
        assign[(p_idx, j_idx)] = var


In [ ]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Desk_ID          942 non-null    object
 1   Req_Grade        942 non-null    int64 
 2   Req_Edu          942 non-null    int64 
 3   Req_Health       942 non-null    int64 
 4   Sec_Clerance     942 non-null    int64 
 5   Req_Appointment  942 non-null    object
dtypes: int64(4), object(2)
memory usage: 44.3+ KB


In [ ]:
#Soft constraints
appointment_terms = []
edu_terms = []

for (p_idx, j_idx), var in assign.items():
    person_apps = people_df.loc[p_idx, "Appointments"]
    job_app = jobs_df.loc[j_idx, "Req_Appointment"]
    
    if job_app in person_apps:
        app_idx = person_apps.index(job_app)
        appoint_score = len(person_apps) - app_idx
    else:
        appoint_score = 0
    appointment_terms.append(var * appoint_score * 5)  # weight (higher the better)

    edu_score = 1 if people_df.loc[p_idx, "Edu_Type"] == jobs_df.loc[j_idx, "Req_Edu"] else 0
    edu_terms.append(var * edu_score * 1) 

In [ ]:
model.Maximize(sum(appointment_terms) + sum(edu_terms) )

solver = cp_model.CpSolver()
status = solver.Solve(model)

In [ ]:
results = []

for (p_idx, j_idx), var in assign.items():
    if solver.BooleanValue(var):
        appoint_match = (
            jobs_df.loc[j_idx, "Req_Appointment"]
            in people_df.loc[p_idx, "Appointments"]
        )
        edu_match = (
            people_df.loc[p_idx, "Edu_Type"]
            == jobs_df.loc[j_idx, "Req_Edu"]
        )

        results.append({
            "Person": people_df.loc[p_idx, "Name"],
            "Job": jobs_df.loc[j_idx, "Desk_ID"],
            "Appointment_Match": int(appoint_match),
            "Edu_Match": int(edu_match),
            "Suitable": int(appoint_match) + int(edu_match)
        })

result_df = pd.DataFrame(results)
print(result_df)

              Person      Job  Appointment_Match  Edu_Match  Suitable
0      Roxuqy Tiwoni  D000051                  0          1         1
1      Roxuqy Tiwoni  D000066                  0          1         1
2      Roxuqy Tiwoni  D000083                  0          1         1
3      Roxuqy Tiwoni  D000166                  0          1         1
4      Roxuqy Tiwoni  D000236                  1          0         1
...              ...      ...                ...        ...       ...
18719  Xideha Kalylu  D000706                  1          0         1
18720  Xideha Kalylu  D000714                  0          1         1
18721  Xideha Kalylu  D000726                  0          1         1
18722  Xideha Kalylu  D000937                  0          1         1
18723  Xideha Kalylu  D000942                  0          1         1

[18724 rows x 5 columns]


In [ ]:
suitable_df = result_df.loc[result_df['Suitable'] == 2]
suitable_df

,Person,Job,Appointment_Match,Edu_Match,Suitable
27,Cagyla Gedyzo,D000818,1,1,2
40,Levyjo Buti,D000278,1,1,2
52,Noro Mamojy,D000076,1,1,2
60,Qecuqu Geseju,D000172,1,1,2
62,Qecuqu Geseju,D000282,1,1,2
...,...,...,...,...,...
18674,Wacyza Nitaca,D000500,1,1,2
18684,Mozuni Ciqe,D000328,1,1,2
18688,Mozuni Ciqe,D000494,1,1,2
18709,Tybufe Cotysy,D000882,1,1,2


In [17]:
result_df.to_excel("Result.xlsx")